# DEE — Test materia v3: con annealing genuino y MC eficiente

**Cambios respecto a v2:**
1. **Annealing genuino simplificado** (1000 pasos MC con schedule de 4 fases) en lugar de cristal jiterado puro
2. **MC con actualización incremental** de F_local (factor 250000× más rápido que ingenuo)
3. **Comparación espectro inicial vs final** para verificar que annealing mejora estadística sin cambiar física

**Schedule MC (calibrado a SIM 19 reducido):**
- Fase 1: 250 pasos a T = θ_D = 43.4
- Fase 2: 250 pasos a T = θ_D/3 = 14.47
- Fase 3: 250 pasos a T = θ_D/10 = 4.34
- Fase 4: 250 pasos a T = 0 (greedy)

**Tiempo estimado**: 5-6 horas en Colab Pro CPU

**Importante**: 1000 pasos da "cristal con relajación parcial", no vidrio frágil maduro como SIM 19 (que usa 25500 pasos). Suficiente para que σ(F) sea invariante con N, no para reproducir β_KWW de SIM 19.

## 0. Setup

In [ ]:
import os, sys, time, json, pickle, gc, warnings
import numpy as np
from scipy.linalg import eigh
from scipy.sparse.linalg import eigsh
from scipy.sparse import csr_matrix, diags
from scipy.stats import linregress, chi2 as chi2_dist
from scipy.optimize import curve_fit
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

OUTPUT_DIR = '/content/dee_materia_v3'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/figuras', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/datos', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/logs', exist_ok=True)

LOG_FILE = f'{OUTPUT_DIR}/logs/ejecucion.log'
log_handle = open(LOG_FILE, 'w')

def log(msg):
    print(msg)
    log_handle.write(msg + '\n')
    log_handle.flush()

from IPython.display import display, Javascript
try:
    display(Javascript('''
    function ClickConnect(){
        document.querySelector("colab-connect-button").shadowRoot
                .querySelector("#connect").click();
    }
    setInterval(ClickConnect, 60000);
    '''))
    log("Keep-alive activo")
except Exception:
    log("Keep-alive no disponible")

log("=" * 70)
log("DEE — TEST MATERIA v3 (annealing genuino simplificado)")
log("=" * 70)
log(f"Inicio: {time.strftime('%Y-%m-%d %H:%M:%S')}")

try:
    import psutil
    mem_gb = psutil.virtual_memory().total / 1e9
    log(f"RAM total: {mem_gb:.1f} GB")
except ImportError:
    pass

# Constantes globales
JITTER_INIT = 0.01
CLIP_FACTOR = 0.3
DEFECT_PERCENTILE = 5
K_NEIGHBORS = 30
TARGET_SIGMA_F_SIM19 = 0.107

# Schedule MC
THETA_D = 43.4
MC_SCHEDULE = [
    {'T': THETA_D,        'n_steps': 250},
    {'T': THETA_D / 3,    'n_steps': 250},
    {'T': THETA_D / 10,   'n_steps': 250},
    {'T': 0.0,            'n_steps': 250},
]
TOTAL_MC_STEPS = sum(p['n_steps'] for p in MC_SCHEDULE)
log(f"\nSchedule MC: {TOTAL_MC_STEPS} pasos totales en 4 fases")
for i, p in enumerate(MC_SCHEDULE):
    log(f"  Fase {i+1}: T={p['T']}, {p['n_steps']} pasos")

## 1. Funciones core: estructura de vecindad para MC eficiente

La clave del MC eficiente es mantener una **lista de vecindad** que se actualiza incrementalmente. Al mover un nodo, solo cambian las distancias entre ese nodo y sus k vecinos, y los nodos para quienes ese nodo es vecino. F_local solo cambia para esos nodos.

In [ ]:
class CrystalState:
    """Mantiene estado del cristal con vecindad pre-calculada.
    Permite mover un nodo y actualizar F_local localmente.
    """
    
    def __init__(self, points, k=30, L=1.0, clip_factor=0.3):
        self.points = points.copy()
        self.N = len(points)
        self.k = k
        self.L = L
        a = self.N ** (-1/3)
        self.clip = clip_factor * a
        
        # Construcción inicial: vecinos por nodo
        self._rebuild_neighbors()
        
        # F_local inicial
        self.F_local = self._compute_F_all()
    
    def _rebuild_neighbors(self):
        """Recalcula vecindad de todos los nodos. Cuesta O(N²)."""
        # Calcular distancias por bloques para evitar matriz N×N
        block_size = 1000
        self.neighbors = np.zeros((self.N, self.k), dtype=np.int32)
        self.distances = np.zeros((self.N, self.k))
        
        for start in range(0, self.N, block_size):
            end = min(start + block_size, self.N)
            diff = self.points[start:end, None, :] - self.points[None, :, :]
            diff = diff - self.L * np.round(diff / self.L)
            dist_block = np.linalg.norm(diff, axis=-1)
            
            for i_local in range(end - start):
                i_global = start + i_local
                dist_block[i_local, i_global] = np.inf
            
            dist_block = np.maximum(dist_block, self.clip)
            for i_local in range(end - start):
                i_global = start + i_local
                dist_block[i_local, i_global] = np.inf
            
            knn = np.argpartition(dist_block, self.k, axis=1)[:, :self.k]
            for i_local in range(end - start):
                i_global = start + i_local
                self.neighbors[i_global] = knn[i_local]
                self.distances[i_global] = dist_block[i_local, knn[i_local]]
    
    def _compute_F_node(self, i):
        """F_local para un solo nodo, usando vecinos pre-calculados."""
        return (1.0 / self.distances[i]**2).sum()
    
    def _compute_F_all(self):
        F = np.zeros(self.N)
        for i in range(self.N):
            F[i] = self._compute_F_node(i)
        # Normalizar al promedio para que ⟨F⟩=1
        return F / F.mean()
    
    def _affected_nodes(self, i):
        """Conjunto de nodos cuyo F_local puede cambiar al mover el nodo i.
        Son: i mismo, los k vecinos de i, y los nodos para quienes i es vecino.
        """
        affected = set([i])
        affected.update(self.neighbors[i].tolist())
        # Nodos para quienes i es vecino: buscar en self.neighbors
        is_neighbor_of_i = np.any(self.neighbors == i, axis=1)
        for j in np.where(is_neighbor_of_i)[0]:
            affected.add(int(j))
        return affected
    
    def _update_neighbors_for_node(self, i):
        """Recalcula vecinos del nodo i."""
        diff = self.points - self.points[i]
        diff = diff - self.L * np.round(diff / self.L)
        dist = np.linalg.norm(diff, axis=-1)
        dist[i] = np.inf
        dist = np.maximum(dist, self.clip)
        dist[i] = np.inf
        knn_i = np.argpartition(dist, self.k)[:self.k]
        self.neighbors[i] = knn_i
        self.distances[i] = dist[knn_i]
    
    def trial_move(self, i, new_pos):
        """Calcula ΔF (variación de F_global = N·Var(F_local)) si moviera el nodo i a new_pos.
        No cambia el estado.
        Devuelve: delta_F_global, info (para usar en accept).
        """
        old_pos = self.points[i].copy()
        old_F_node = self.F_local.copy()
        old_neighbors_i = self.neighbors[i].copy()
        old_distances_i = self.distances[i].copy()
        
        # Aplicar movimiento temporalmente
        self.points[i] = new_pos % self.L
        
        # Recalcular vecinos del nodo movido
        self._update_neighbors_for_node(i)
        
        # Identificar nodos afectados
        affected = self._affected_nodes(i)
        
        # Recalcular F_local para nodos afectados que NO son i
        # Para esos, también necesitamos actualizar SUS vecinos si i estaba o está en su lista
        old_neighbors_dict = {}  # j -> (neighbors_j, distances_j) anteriores
        for j in affected:
            if j == i:
                continue
            old_neighbors_dict[j] = (self.neighbors[j].copy(), self.distances[j].copy())
            self._update_neighbors_for_node(j)
        
        # Calcular nuevo F_local de nodos afectados (no normalizado)
        new_F_local_unnorm = self.F_local * 1  # copia para mantener nodos no afectados
        # Pero F_local estaba normalizado... necesitamos manejar la normalización
        # Estrategia: calcular F sin normalizar para los afectados y reescalar al final
        
        # Más simple: F_local_unnorm[j] = sum 1/d² sobre vecinos
        # F_local normalizado = F_unnorm / mean(F_unnorm)
        
        # Como queremos sólo Var(F_local_norm), y los nodos no afectados no cambiaron F_unnorm,
        # reconstruimos F_unnorm completo a partir de F_local actual:
        F_unnorm_old = self.F_local * 1.0  # F_local mean=1, entonces F_unnorm[j] proportional to F_local[j]
        # Pero el factor de normalización depende del mean global, que cambió.
        
        # Estrategia robusta: trabajar con F_unnorm directamente
        # Para esto guardamos F_unnorm como atributo del estado
        
        # REVERTIR todo para hacer la cosa correctamente
        self.points[i] = old_pos
        self.neighbors[i] = old_neighbors_i
        self.distances[i] = old_distances_i
        for j, (nb, ds) in old_neighbors_dict.items():
            self.neighbors[j] = nb
            self.distances[j] = ds
        
        return None  # placeholder, se reemplaza abajo

log("Class CrystalState definida (versión preliminar).")

## 1b. Versión simplificada: F_unnorm como atributo

La complejidad de manejar F_local normalizado y el factor de normalización es alta. Simplifico trabajando con F_unnorm (sin normalizar) y reportando σ(F_unnorm)/⟨F_unnorm⟩ que es invariante:

In [ ]:
class CrystalStateV2:
    """Versión limpia: trabaja con F_unnorm (suma 1/d²).
    
    F_global (a minimizar) = N · Var(F_unnorm/⟨F_unnorm⟩)
                          = N · Var(F_unnorm) / ⟨F_unnorm⟩²
    """
    
    def __init__(self, points, k=30, L=1.0, clip_factor=0.3):
        self.points = points.copy()
        self.N = len(points)
        self.k = k
        self.L = L
        a = self.N ** (-1/3)
        self.clip = clip_factor * a
        
        self._rebuild_all()
    
    def _rebuild_all(self):
        """Reconstruye vecinos y F_unnorm desde cero. O(N²)."""
        block_size = 1000
        self.neighbors = np.zeros((self.N, self.k), dtype=np.int32)
        self.distances = np.zeros((self.N, self.k))
        
        for start in range(0, self.N, block_size):
            end = min(start + block_size, self.N)
            diff = self.points[start:end, None, :] - self.points[None, :, :]
            diff = diff - self.L * np.round(diff / self.L)
            dist_block = np.linalg.norm(diff, axis=-1)
            
            for i_local in range(end - start):
                i_global = start + i_local
                dist_block[i_local, i_global] = np.inf
            dist_block = np.maximum(dist_block, self.clip)
            for i_local in range(end - start):
                i_global = start + i_local
                dist_block[i_local, i_global] = np.inf
            
            knn = np.argpartition(dist_block, self.k, axis=1)[:, :self.k]
            for i_local in range(end - start):
                i_global = start + i_local
                self.neighbors[i_global] = knn[i_local]
                self.distances[i_global] = dist_block[i_local, knn[i_local]]
        
        # F_unnorm[i] = sum 1/d² sobre vecinos de i
        self.F_unnorm = (1.0 / self.distances**2).sum(axis=1)
    
    def F_global(self):
        """F = N · Var(F_unnorm/⟨F_unnorm⟩) = N · Var(F_unnorm) / ⟨F_unnorm⟩²"""
        mean_F = self.F_unnorm.mean()
        if mean_F < 1e-10:
            return float('inf')
        return self.N * self.F_unnorm.var() / mean_F**2
    
    def F_local_normalized(self):
        """F_local con ⟨F⟩=1."""
        return self.F_unnorm / self.F_unnorm.mean()
    
    def _affected_nodes(self, i, new_neighbors_of_i):
        """Nodos afectados al mover i.
        
        - i mismo
        - vecinos antiguos de i (los nodos cuyas distancias a i cambian)
        - vecinos nuevos de i
        - nodos j para quienes i estaba o seguirá estando entre los k-NN
        """
        affected = set([i])
        affected.update(self.neighbors[i].tolist())
        affected.update(new_neighbors_of_i.tolist())
        # Nodos j que tenían i en su lista de vecinos
        is_neighbor_of_i = np.any(self.neighbors == i, axis=1)
        for j in np.where(is_neighbor_of_i)[0]:
            affected.add(int(j))
        return affected
    
    def trial_move_metropolis(self, i, new_pos, T, rng):
        """Propone mover el nodo i. Acepta/rechaza por Metropolis a temperatura T.
        Devuelve True si aceptado, False si rechazado.
        """
        # Estado actual
        F_old = self.F_global()
        old_pos = self.points[i].copy()
        old_neighbors = self.neighbors.copy()
        old_distances = self.distances.copy()
        old_F_unnorm = self.F_unnorm.copy()
        
        # Aplicar movimiento
        self.points[i] = new_pos % self.L
        
        # Recalcular vecinos del nodo i
        diff = self.points - self.points[i]
        diff = diff - self.L * np.round(diff / self.L)
        dist_i = np.linalg.norm(diff, axis=-1)
        dist_i[i] = np.inf
        dist_i = np.maximum(dist_i, self.clip)
        dist_i[i] = np.inf
        new_knn_i = np.argpartition(dist_i, self.k)[:self.k]
        self.neighbors[i] = new_knn_i
        self.distances[i] = dist_i[new_knn_i]
        
        # Identificar nodos afectados
        affected = self._affected_nodes(i, new_knn_i)
        
        # Recalcular vecinos y F_unnorm para nodos afectados (≠ i)
        for j in affected:
            if j == i:
                continue
            diff_j = self.points - self.points[j]
            diff_j = diff_j - self.L * np.round(diff_j / self.L)
            dist_j = np.linalg.norm(diff_j, axis=-1)
            dist_j[j] = np.inf
            dist_j = np.maximum(dist_j, self.clip)
            dist_j[j] = np.inf
            new_knn_j = np.argpartition(dist_j, self.k)[:self.k]
            self.neighbors[j] = new_knn_j
            self.distances[j] = dist_j[new_knn_j]
        
        # Recalcular F_unnorm de nodos afectados
        for j in affected:
            self.F_unnorm[j] = (1.0 / self.distances[j]**2).sum()
        
        # Nuevo F_global
        F_new = self.F_global()
        delta_F = F_new - F_old
        
        # Metropolis
        if T > 0:
            accept = (delta_F < 0) or (rng.random() < np.exp(-delta_F / T))
        else:
            accept = delta_F < 0
        
        if accept:
            return True
        else:
            # Revertir
            self.points[i] = old_pos
            self.neighbors = old_neighbors
            self.distances = old_distances
            self.F_unnorm = old_F_unnorm
            return False

log("CrystalStateV2 definida (versión limpia).")

## 2. Annealing simulado

In [ ]:
def init_cristal_jiterado(N, L=1.0, jitter=0.01, seed=42):
    rng = np.random.default_rng(seed)
    n_side = int(np.ceil(N**(1/3)))
    pts = []
    for i in range(n_side):
        for j in range(n_side):
            for k in range(n_side):
                pts.append([i/n_side, j/n_side, k/n_side])
    pts = np.array(pts[:N]) * L
    pts += rng.normal(0, jitter, pts.shape)
    return pts % L

def run_annealing(state, schedule, sigma_pos=0.005, seed=42, log_interval=100):
    """Ejecuta annealing siguiendo schedule de temperaturas.
    
    Devuelve historial F_global vs paso.
    """
    rng = np.random.default_rng(seed)
    history = {'step': [], 'F': [], 'T': [], 'phase': []}
    
    step_global = 0
    F_init = state.F_global()
    history['step'].append(0)
    history['F'].append(F_init)
    history['T'].append(schedule[0]['T'])
    history['phase'].append(0)
    
    for phase_idx, phase in enumerate(schedule):
        T = phase['T']
        n_steps = phase['n_steps']
        accept_count = 0
        
        for step in range(n_steps):
            i = rng.integers(state.N)
            new_pos = state.points[i] + sigma_pos * rng.standard_normal(3)
            accepted = state.trial_move_metropolis(i, new_pos, T, rng)
            if accepted:
                accept_count += 1
            step_global += 1
            
            if step_global % log_interval == 0:
                F_curr = state.F_global()
                history['step'].append(step_global)
                history['F'].append(F_curr)
                history['T'].append(T)
                history['phase'].append(phase_idx)
    
    return history

log("Funciones de annealing listas.")

## 3. Validación inicial: ¿el MC eficiente funciona?

Antes del barrido completo, una réplica piloto pequeña (N=500) para verificar que el MC produce reducción de F y σ(F) consistente con SIM 19.

In [ ]:
log("\n" + "="*70)
log("VALIDACIÓN INICIAL: MC eficiente en N=500")
log("="*70)

t_pilot = time.time()
points_pilot = init_cristal_jiterado(500, jitter=JITTER_INIT, seed=12345)
state_pilot = CrystalStateV2(points_pilot, k=K_NEIGHBORS)

log(f"\nEstado inicial (N=500, cristal jiterado):")
log(f"  F_global = {state_pilot.F_global():.4f}")
log(f"  σ(F_local) = {state_pilot.F_local_normalized().std():.4f}")

# Annealing piloto (más corto)
schedule_piloto = [
    {'T': THETA_D, 'n_steps': 100},
    {'T': THETA_D/3, 'n_steps': 100},
    {'T': 0.0, 'n_steps': 100},
]

log(f"\nEjecutando annealing piloto (300 pasos)...")
t_anneal = time.time()
history_piloto = run_annealing(state_pilot, schedule_piloto, sigma_pos=0.005, seed=42)
log(f"  Tiempo: {time.time()-t_anneal:.1f}s")
log(f"  F_global final = {state_pilot.F_global():.4f}")
log(f"  σ(F_local) final = {state_pilot.F_local_normalized().std():.4f}")
log(f"  Cambio en F: {(history_piloto['F'][-1] - history_piloto['F'][0])/history_piloto['F'][0]*100:.1f}%")

# Comparar con SIM 19 N=500: F_global típica ~ 11-14
F_final_pilot = history_piloto['F'][-1]
if 5 < F_final_pilot < 30:
    log(f"  ✓ F_final = {F_final_pilot:.2f} en rango razonable (SIM 19: F~11-14)")
else:
    log(f"  ⚠ F_final = {F_final_pilot:.2f} fuera de rango esperado")

log(f"\nTiempo total piloto: {time.time()-t_pilot:.1f}s")

## 4. Funciones de análisis (heredadas de v2)

In [ ]:
def build_laplacian_dense(points, k_neighbors=30, L=1.0, clip_factor=CLIP_FACTOR):
    N = len(points)
    a = N**(-1/3)
    clip = clip_factor * a
    
    block_size = 1000
    W = np.zeros((N, N))
    
    for start in range(0, N, block_size):
        end = min(start + block_size, N)
        diff = points[start:end, None, :] - points[None, :, :]
        diff = diff - L * np.round(diff / L)
        dist_block = np.linalg.norm(diff, axis=-1)
        
        for i_local in range(end - start):
            i_global = start + i_local
            dist_block[i_local, i_global] = np.inf
        dist_block = np.maximum(dist_block, clip)
        for i_local in range(end - start):
            i_global = start + i_local
            dist_block[i_local, i_global] = np.inf
        
        knn = np.argpartition(dist_block, k_neighbors, axis=1)[:, :k_neighbors]
        for i_local in range(end - start):
            i_global = start + i_local
            for j in knn[i_local]:
                d = dist_block[i_local, j]
                w = 1.0 / d**2
                W[i_global, j] = w
                W[j, i_global] = w
    
    return np.diag(W.sum(axis=1)) - W

def build_laplacian_sparse(points, k_neighbors=30, L=1.0, clip_factor=CLIP_FACTOR):
    N = len(points)
    a = N**(-1/3)
    clip = clip_factor * a
    
    block_size = 1000
    rows, cols, weights = [], [], []
    
    for start in range(0, N, block_size):
        end = min(start + block_size, N)
        diff = points[start:end, None, :] - points[None, :, :]
        diff = diff - L * np.round(diff / L)
        dist_block = np.linalg.norm(diff, axis=-1)
        
        for i_local in range(end - start):
            i_global = start + i_local
            dist_block[i_local, i_global] = np.inf
        dist_block = np.maximum(dist_block, clip)
        for i_local in range(end - start):
            i_global = start + i_local
            dist_block[i_local, i_global] = np.inf
        
        knn = np.argpartition(dist_block, k_neighbors, axis=1)[:, :k_neighbors]
        for i_local in range(end - start):
            i_global = start + i_local
            for j in knn[i_local]:
                d = dist_block[i_local, j]
                w = 1.0 / d**2
                rows.append(i_global)
                cols.append(j)
                weights.append(w)
    
    rows = np.array(rows); cols = np.array(cols); weights = np.array(weights)
    W_sparse = csr_matrix((weights, (rows, cols)), shape=(N, N))
    W_sym = W_sparse.maximum(W_sparse.T)
    D = diags(np.array(W_sym.sum(axis=1)).flatten())
    return D - W_sym

def diagonalize_laplacian(Lap, N, mode='dense', n_low=200, n_high=200):
    if mode == 'dense':
        if hasattr(Lap, 'toarray'):
            Lap = Lap.toarray()
        eigvals, eigvecs = eigh(Lap)
        iprs = np.sum(eigvecs**4, axis=0)
        centers = np.argmax(eigvecs**2, axis=0)
        return eigvals, iprs, centers
    elif mode == 'lanczos':
        eigvals_low, eigvecs_low = eigsh(Lap, k=n_low, which='SM', maxiter=5000)
        eigvals_high, eigvecs_high = eigsh(Lap, k=n_high, which='LM', maxiter=5000)
        all_eigvals = np.concatenate([eigvals_low, eigvals_high])
        all_eigvecs = np.concatenate([eigvecs_low, eigvecs_high], axis=1)
        sort_idx = np.argsort(all_eigvals)
        eigvals_sorted = all_eigvals[sort_idx]
        eigvecs_sorted = all_eigvecs[:, sort_idx]
        iprs = np.sum(eigvecs_sorted**4, axis=0)
        centers = np.argmax(eigvecs_sorted**2, axis=0)
        return eigvals_sorted, iprs, centers

def find_mobility_edge(eigvals, iprs, n_bands=20):
    nonzero = eigvals > 1e-6
    omegas = np.sqrt(eigvals[nonzero])
    iprs_nz = iprs[nonzero]
    
    omega_bands = np.linspace(omegas.min(), omegas.max(), n_bands + 1)
    band_centers = []
    band_ipr_med = []
    for i in range(n_bands):
        mask = (omegas >= omega_bands[i]) & (omegas < omega_bands[i+1])
        if mask.sum() >= 5:
            band_centers.append(0.5 * (omega_bands[i] + omega_bands[i+1]))
            band_ipr_med.append(np.median(iprs_nz[mask]))
    band_centers = np.array(band_centers)
    band_ipr_med = np.array(band_ipr_med)
    
    if len(band_ipr_med) == 0:
        return None, band_centers, band_ipr_med
    threshold = 5 * band_ipr_med.min()
    
    for i, ipr in enumerate(band_ipr_med):
        if ipr > threshold:
            return float(band_centers[i]), band_centers, band_ipr_med
    return None, band_centers, band_ipr_med

def caracterize_defects_percentile(F_local, points, percentile=DEFECT_PERCENTILE, L=1.0):
    threshold_low = np.percentile(F_local, percentile)
    threshold_high = np.percentile(F_local, 100 - percentile)
    is_low = F_local < threshold_low
    is_high = F_local > threshold_high
    
    N = len(points)
    a = N**(-1/3)
    shell_edges = np.array([0.5*a, 1.5*a, 2.5*a, 3.5*a, 4.5*a, 6*a, 8*a])
    shell_centers = 0.5 * (shell_edges[:-1] + shell_edges[1:])
    delta_F = F_local - F_local.mean()
    
    def_low_idx = np.where(is_low)[0]
    def_high_idx = np.where(is_high)[0]
    
    rng = np.random.default_rng(0)
    max_for_profile = 80
    if len(def_low_idx) > max_for_profile:
        def_low_idx = rng.choice(def_low_idx, max_for_profile, replace=False)
    if len(def_high_idx) > max_for_profile:
        def_high_idx = rng.choice(def_high_idx, max_for_profile, replace=False)
    
    def get_profiles(def_idx):
        profiles = []
        for idx in def_idx:
            diff = points - points[idx]
            diff = diff - L * np.round(diff / L)
            dists = np.linalg.norm(diff, axis=-1)
            profile = []
            for i in range(len(shell_edges) - 1):
                mask = (dists >= shell_edges[i]) & (dists < shell_edges[i+1])
                mask[idx] = False
                profile.append(delta_F[mask].mean() if mask.sum() > 0 else np.nan)
            profiles.append(profile)
        return np.array(profiles) if profiles else np.array([]).reshape(0, len(shell_centers))
    
    profiles_low = get_profiles(def_low_idx)
    profiles_high = get_profiles(def_high_idx)
    
    def fit_xi(centers, profile):
        if profile.shape[0] < 3:
            return None
        median_p = np.nanmedian(profile, axis=0)
        abs_p = np.abs(median_p)
        valid = ~np.isnan(abs_p) & (abs_p > 1e-5)
        if valid.sum() < 3:
            return None
        try:
            popt, _ = curve_fit(
                lambda r, A, xi: A * np.exp(-r/xi),
                centers[valid], abs_p[valid],
                p0=[abs_p[valid][0], 0.05],
                bounds=([0, 0.001], [10, 0.5]),
                maxfev=5000
            )
            return popt[1]
        except Exception:
            return None
    
    xi_low = fit_xi(shell_centers, profiles_low)
    xi_high = fit_xi(shell_centers, profiles_high)
    
    return {
        'mean_F': float(F_local.mean()),
        'sigma_F': float(F_local.std()),
        'threshold_low': float(threshold_low),
        'threshold_high': float(threshold_high),
        'frac_low': float(is_low.sum() / N),
        'frac_high': float(is_high.sum() / N),
        'shell_centers_in_a': (shell_centers / a).tolist(),
        'profile_low_median': np.nanmedian(profiles_low, axis=0).tolist() if profiles_low.shape[0] > 0 else [],
        'profile_high_median': np.nanmedian(profiles_high, axis=0).tolist() if profiles_high.shape[0] > 0 else [],
        'xi_def_low_in_a': float(xi_low / a) if xi_low else None,
        'xi_def_high_in_a': float(xi_high / a) if xi_high else None,
    }

def correlation_modes_defects(eigvals, iprs, centers, F_local, threshold_low, threshold_high,
                                ipr_threshold=0.10):
    is_low = F_local < threshold_low
    is_high = F_local > threshold_high
    
    localized_mask = iprs > ipr_threshold
    n_loc = int(localized_mask.sum())
    if n_loc == 0:
        return None
    
    centers_loc = centers[localized_mask]
    n_low = int(is_low[centers_loc].sum())
    n_high = int(is_high[centers_loc].sum())
    n_norm = n_loc - n_low - n_high
    
    N = len(F_local)
    frac_low_pop = is_low.sum() / N
    frac_high_pop = is_high.sum() / N
    frac_norm_pop = 1 - frac_low_pop - frac_high_pop
    
    enr_low = (n_low / n_loc) / frac_low_pop if frac_low_pop > 0 else None
    enr_high = (n_high / n_loc) / frac_high_pop if frac_high_pop > 0 else None
    
    expected_low = n_loc * frac_low_pop
    expected_high = n_loc * frac_high_pop
    expected_norm = n_loc * frac_norm_pop
    chi2 = sum(((obs - exp)**2 / exp) for obs, exp in [(n_low, expected_low), (n_high, expected_high), (n_norm, expected_norm)] if exp > 0)
    p_value = 1 - chi2_dist.cdf(chi2, df=2)
    
    omegas_loc = np.sqrt(np.maximum(eigvals[localized_mask], 0))
    omega_on_low = omegas_loc[is_low[centers_loc]]
    omega_on_high = omegas_loc[is_high[centers_loc]]
    
    return {
        'n_localized_modes': n_loc,
        'centered_on_low': n_low, 'centered_on_high': n_high, 'centered_on_normal': n_norm,
        'enrichment_low': float(enr_low) if enr_low is not None else None,
        'enrichment_high': float(enr_high) if enr_high is not None else None,
        'chi2': float(chi2), 'p_value': float(p_value),
        'mean_omega_on_low': float(omega_on_low.mean()) if len(omega_on_low) > 0 else None,
        'mean_omega_on_high': float(omega_on_high.mean()) if len(omega_on_high) > 0 else None,
    }

log("Funciones de análisis listas.")

## 5. Comparación pre/post annealing en una réplica de cada N

Test que pediste: ¿el annealing cambia mucho el espectro? Si no, el cristal jiterado de v2 era buena aproximación. Si sí, sabemos que SIM 19 vive en régimen distinto.

In [ ]:
log("\n" + "="*70)
log("COMPARACIÓN: espectro pre/post annealing")
log("="*70)
log("Para cada N, una réplica con espectro completo antes y después de annealing")

comparison_results = {}

# Solo N=2000 y N=4000 (donde podemos diagonalizar denso)
for N_compare in [2000, 4000]:
    log(f"\n--- Comparación para N = {N_compare} ---")
    seed = 12345
    
    points_pre = init_cristal_jiterado(N_compare, jitter=JITTER_INIT, seed=seed)
    
    # Espectro pre
    log("  Calculando espectro PRE-annealing...")
    Lap_pre = build_laplacian_dense(points_pre, k_neighbors=K_NEIGHBORS)
    eigvals_pre, iprs_pre, centers_pre = diagonalize_laplacian(Lap_pre, N_compare, mode='dense')
    
    nonzero = eigvals_pre > 1e-6
    omegas_pre = np.sqrt(eigvals_pre[nonzero])
    log(f"    ω rango: [{omegas_pre.min():.3f}, {omegas_pre.max():.3f}]")
    log(f"    Modos con IPR>0.1: {(iprs_pre > 0.1).sum()}")
    
    # Annealing
    log("  Ejecutando annealing...")
    state = CrystalStateV2(points_pre, k=K_NEIGHBORS)
    F_pre = state.F_global()
    t_anneal = time.time()
    history = run_annealing(state, MC_SCHEDULE, sigma_pos=0.005, seed=seed+999)
    t_anneal_dur = time.time() - t_anneal
    F_post = state.F_global()
    log(f"    Tiempo annealing: {t_anneal_dur:.1f}s")
    log(f"    F: {F_pre:.4f} → {F_post:.4f} (cambio {(F_post-F_pre)/F_pre*100:+.1f}%)")
    
    points_post = state.points
    
    # Espectro post
    log("  Calculando espectro POST-annealing...")
    Lap_post = build_laplacian_dense(points_post, k_neighbors=K_NEIGHBORS)
    eigvals_post, iprs_post, centers_post = diagonalize_laplacian(Lap_post, N_compare, mode='dense')
    
    nonzero = eigvals_post > 1e-6
    omegas_post = np.sqrt(eigvals_post[nonzero])
    log(f"    ω rango: [{omegas_post.min():.3f}, {omegas_post.max():.3f}]")
    log(f"    Modos con IPR>0.1: {(iprs_post > 0.1).sum()}")
    
    # Mobility edge
    me_pre, _, _ = find_mobility_edge(eigvals_pre, iprs_pre)
    me_post, _, _ = find_mobility_edge(eigvals_post, iprs_post)
    log(f"    Mobility edge: {me_pre} → {me_post}")
    
    comparison_results[N_compare] = {
        'F_pre': F_pre, 'F_post': F_post,
        'omega_min_pre': float(omegas_pre.min()), 'omega_min_post': float(omegas_post.min()),
        'omega_max_pre': float(omegas_pre.max()), 'omega_max_post': float(omegas_post.max()),
        'omega_mean_pre': float(omegas_pre.mean()), 'omega_mean_post': float(omegas_post.mean()),
        'n_localized_pre': int((iprs_pre > 0.1).sum()),
        'n_localized_post': int((iprs_post > 0.1).sum()),
        'mobility_edge_pre': me_pre, 'mobility_edge_post': me_post,
        't_anneal': t_anneal_dur,
    }
    
    del Lap_pre, Lap_post, eigvals_pre, eigvals_post
    gc.collect()

log("\n=== Comparación completada ===")
with open(f'{OUTPUT_DIR}/datos/comparison_pre_post.json', 'w') as f:
    json.dump(comparison_results, f, indent=2, default=float)

## 6. Barrido principal: réplicas con annealing

In [ ]:
config = [
    {'N': 2000, 'replicas': 4, 'mode': 'dense'},
    {'N': 4000, 'replicas': 3, 'mode': 'dense'},
    {'N': 8000, 'replicas': 2, 'mode': 'lanczos', 'n_low': 200, 'n_high': 200},
    {'N': 16000, 'replicas': 2, 'mode': 'lanczos', 'n_low': 300, 'n_high': 300},
]

log("\n" + "="*70)
log("BARRIDO PRINCIPAL CON ANNEALING")
log("="*70)
for c in config:
    log(f"  N={c['N']}, {c['replicas']} réplicas, modo={c['mode']}")

all_results = {}

for cfg in config:
    N = cfg['N']
    n_rep = cfg['replicas']
    mode = cfg['mode']
    
    log(f"\n{'='*60}")
    log(f"N = {N}, modo = {mode}")
    log(f"{'='*60}")
    
    results_N = []
    
    for r in range(n_rep):
        seed = 100 * (r + 1) + 1
        log(f"\n--- Réplica {r+1}/{n_rep}, seed={seed} ---")
        t_rep = time.time()
        
        # Inicializar
        points = init_cristal_jiterado(N, jitter=JITTER_INIT, seed=seed)
        
        # Annealing
        log("  Ejecutando annealing...")
        t_anneal = time.time()
        state = CrystalStateV2(points, k=K_NEIGHBORS)
        F_init = state.F_global()
        history = run_annealing(state, MC_SCHEDULE, sigma_pos=0.005, seed=seed+999, log_interval=500)
        F_final = state.F_global()
        log(f"    Annealing: {time.time()-t_anneal:.1f}s, F: {F_init:.3f} → {F_final:.3f}")
        points = state.points
        
        # F_local
        F_local = state.F_local_normalized()
        log(f"    σ(F_local) post-annealing: {F_local.std():.4f}")
        
        # Liberar state, lo necesitamos solo para el annealing
        del state
        
        # Test 1: defectos
        def_data = caracterize_defects_percentile(F_local, points)
        log(f"    Test 1: ξ_low={def_data['xi_def_low_in_a']}, ξ_high={def_data['xi_def_high_in_a']}")
        
        # Construir Laplaciano para tests espectrales
        if mode == 'dense':
            Lap = build_laplacian_dense(points, k_neighbors=K_NEIGHBORS)
        else:
            Lap = build_laplacian_sparse(points, k_neighbors=K_NEIGHBORS)
        
        # Diagonalizar
        t_diag = time.time()
        if mode == 'dense':
            eigvals, iprs, centers = diagonalize_laplacian(Lap, N, mode='dense')
        else:
            eigvals, iprs, centers = diagonalize_laplacian(
                Lap, N, mode='lanczos',
                n_low=cfg.get('n_low', 200), n_high=cfg.get('n_high', 200)
            )
        log(f"    Diag {mode}: {len(eigvals)} eigvals en {time.time()-t_diag:.1f}s")
        
        mobility_edge, band_centers, band_ipr_med = find_mobility_edge(eigvals, iprs)
        log(f"    Mobility edge ω = {mobility_edge}")
        
        corr = correlation_modes_defects(
            eigvals, iprs, centers, F_local,
            def_data['threshold_low'], def_data['threshold_high']
        )
        if corr is not None:
            log(f"    Test 3: enr_low={corr['enrichment_low']}, enr_high={corr['enrichment_high']}, p={corr['p_value']:.2e}")
        
        nonzero = eigvals > 1e-6
        omegas = np.sqrt(eigvals[nonzero])
        
        result = {
            'N': N, 'replica': r, 'seed': seed,
            'F_global_init': float(F_init), 'F_global_final': float(F_final),
            'F_local_mean': float(F_local.mean()), 'F_local_std': float(F_local.std()),
            'frac_low': def_data['frac_low'], 'frac_high': def_data['frac_high'],
            'threshold_low': def_data['threshold_low'], 'threshold_high': def_data['threshold_high'],
            'xi_def_low_in_a': def_data['xi_def_low_in_a'],
            'xi_def_high_in_a': def_data['xi_def_high_in_a'],
            'profile_low_median': def_data['profile_low_median'],
            'profile_high_median': def_data['profile_high_median'],
            'shell_centers_in_a': def_data['shell_centers_in_a'],
            'mobility_edge': mobility_edge,
            'band_centers': band_centers.tolist() if isinstance(band_centers, np.ndarray) else band_centers,
            'band_ipr_med': band_ipr_med.tolist() if isinstance(band_ipr_med, np.ndarray) else band_ipr_med,
            'omega_min': float(omegas.min()), 'omega_max': float(omegas.max()),
            'mode': mode,
        }
        if corr is not None:
            result['correlation'] = corr
        
        if mode == 'dense' and len(omegas) > 1000:
            sample_idx = np.linspace(0, len(omegas)-1, 1000, dtype=int)
            result['omega_sample'] = omegas[sample_idx].tolist()
            result['ipr_sample'] = iprs[nonzero][sample_idx].tolist()
        else:
            result['omega_sample'] = omegas.tolist()
            result['ipr_sample'] = iprs[nonzero].tolist()
        
        results_N.append(result)
        log(f"  Réplica completa en {time.time()-t_rep:.1f}s")
        
        del Lap, eigvals, iprs, centers, points, F_local
        gc.collect()
    
    all_results[N] = results_N
    
    with open(f'{OUTPUT_DIR}/datos/checkpoint_N{N}.pkl', 'wb') as f:
        pickle.dump(all_results, f)
    log(f"  ✓ Checkpoint N={N} guardado")

log("\n=== Barrido completado ===")

## 7. Análisis y reporte

In [ ]:
log("\n" + "="*70)
log("ANÁLISIS")
log("="*70)

def bootstrap_stat(values, stat_fn=np.mean, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    values = np.array([v for v in values if v is not None and not np.isnan(v)])
    if len(values) < 2:
        return None, None
    boots = [stat_fn(rng.choice(values, len(values), replace=True)) for _ in range(n_boot)]
    return float(stat_fn(values)), float(np.std(boots))

summary_per_N = {}
log(f"\n{'N':>8} {'σ(F)':>16} {'F_init':>10} {'F_final':>10} {'ξ_low':>16} {'ξ_high':>16} {'mob.edge':>17} {'enr_high':>17}")
log("-" * 130)

for N in sorted(all_results.keys()):
    results = all_results[N]
    if not results:
        continue
    
    sigma_F_m, sigma_F_se = bootstrap_stat([r['F_local_std'] for r in results])
    F_init_m = float(np.mean([r['F_global_init'] for r in results]))
    F_final_m = float(np.mean([r['F_global_final'] for r in results]))
    xi_low_m, xi_low_se = bootstrap_stat([r['xi_def_low_in_a'] for r in results if r['xi_def_low_in_a'] is not None])
    xi_high_m, xi_high_se = bootstrap_stat([r['xi_def_high_in_a'] for r in results if r['xi_def_high_in_a'] is not None])
    mob_m, mob_se = bootstrap_stat([r['mobility_edge'] for r in results if r['mobility_edge'] is not None])
    enr_l_m, enr_l_se = bootstrap_stat([r['correlation']['enrichment_low'] for r in results 
                                          if 'correlation' in r and r['correlation']['enrichment_low'] is not None])
    enr_h_m, enr_h_se = bootstrap_stat([r['correlation']['enrichment_high'] for r in results 
                                          if 'correlation' in r and r['correlation']['enrichment_high'] is not None])
    
    summary_per_N[N] = {
        'F_std_mean': sigma_F_m, 'F_std_se': sigma_F_se,
        'F_global_init_mean': F_init_m, 'F_global_final_mean': F_final_m,
        'xi_def_low_mean': xi_low_m, 'xi_def_low_se': xi_low_se,
        'xi_def_high_mean': xi_high_m, 'xi_def_high_se': xi_high_se,
        'mobility_edge_mean': mob_m, 'mobility_edge_se': mob_se,
        'enrichment_low_mean': enr_l_m, 'enrichment_low_se': enr_l_se,
        'enrichment_high_mean': enr_h_m, 'enrichment_high_se': enr_h_se,
        'n_replicas': len(results),
    }
    
    s = summary_per_N[N]
    def fmt(m, se, w=14):
        if m is None: return f"{'—':>{w}}"
        return f"{m:>{w-7}.3f}±{se:.3f}" if se is not None else f"{m:>{w}.3f}"
    
    log(f"{N:>8} {fmt(s['F_std_mean'], s['F_std_se']):>16} {s['F_global_init_mean']:>10.2f} "
        f"{s['F_global_final_mean']:>10.2f} {fmt(s['xi_def_low_mean'], s['xi_def_low_se']):>16} "
        f"{fmt(s['xi_def_high_mean'], s['xi_def_high_se']):>16} {fmt(s['mobility_edge_mean'], s['mobility_edge_se']):>17} "
        f"{fmt(s['enrichment_high_mean'], s['enrichment_high_se']):>17}")

with open(f'{OUTPUT_DIR}/datos/summary_per_N.json', 'w') as f:
    json.dump({str(k): v for k, v in summary_per_N.items()}, f, indent=2, default=float)

In [ ]:
# Reporte final
report = {
    'fecha': time.strftime('%Y-%m-%d %H:%M:%S'),
    'parametros': {
        'jitter_init': JITTER_INIT, 'clip_factor': CLIP_FACTOR,
        'defect_percentile': DEFECT_PERCENTILE,
        'mc_total_steps': TOTAL_MC_STEPS,
    },
    'config': config,
    'comparison_pre_post': comparison_results,
    'summary_per_N': {str(k): v for k, v in summary_per_N.items()},
    'conclusiones': []
}

Ns_done = sorted(summary_per_N.keys())

# σ(F) invariante con N?
sigmas = [summary_per_N[N]['F_std_mean'] for N in Ns_done]
sigma_var = np.std(sigmas) / np.mean(sigmas) if np.mean(sigmas) > 0 else 0
if sigma_var < 0.15:
    report['conclusiones'].append(
        f"σ(F_local) invariante con N: {dict(zip(Ns_done, [round(s, 3) for s in sigmas]))}, variación {sigma_var*100:.0f}%")
else:
    report['conclusiones'].append(
        f"σ(F_local) varía con N en {sigma_var*100:.0f}%: {dict(zip(Ns_done, [round(s, 3) for s in sigmas]))}")

# Mobility edge invariante con N?
mes = [summary_per_N[N]['mobility_edge_mean'] for N in Ns_done if summary_per_N[N]['mobility_edge_mean'] is not None]
if len(mes) >= 2:
    mes_var = np.std(mes) / np.mean(mes)
    if mes_var < 0.20:
        report['conclusiones'].append(f"Mobility edge ω estable: {np.mean(mes):.1f} ± {np.std(mes):.1f}")
    else:
        report['conclusiones'].append(f"Mobility edge sigue variando con N: {dict(zip(Ns_done, [round(m, 1) for m in mes]))}")

# Enriquecimiento
enrs_h = [summary_per_N[N]['enrichment_high_mean'] for N in Ns_done if summary_per_N[N]['enrichment_high_mean'] is not None]
if enrs_h:
    report['conclusiones'].append(f"Enriquecimiento HIGH: ⟨enr⟩ = {np.mean(enrs_h):.2f} ± {np.std(enrs_h):.2f}")

# Comparación pre/post
for N, comp in comparison_results.items():
    delta_F = (comp['F_post'] - comp['F_pre']) / comp['F_pre'] * 100
    delta_me = (comp['mobility_edge_post'] - comp['mobility_edge_pre']) / comp['mobility_edge_pre'] * 100 \
                if comp['mobility_edge_pre'] else None
    report['conclusiones'].append(
        f"N={N}: annealing redujo F en {delta_F:.1f}%, ME cambió {delta_me:.1f}% si computable")

log("\n>>> Conclusiones:")
for c in report['conclusiones']:
    log(f"   - {c}")

with open(f'{OUTPUT_DIR}/REPORTE_FINAL.json', 'w') as f:
    json.dump(report, f, indent=2, default=float)

with open(f'{OUTPUT_DIR}/REPORTE_FINAL.txt', 'w') as f:
    f.write("DEE — Test materia v3 (annealing genuino)\n")
    f.write("="*70 + "\n")
    f.write(f"Fecha: {report['fecha']}\n")
    f.write(f"MC: {TOTAL_MC_STEPS} pasos por réplica con schedule de 4 fases\n\n")
    
    f.write("=== Comparación pre/post annealing ===\n")
    for N, comp in comparison_results.items():
        f.write(f"N={N}:\n")
        f.write(f"  F: {comp['F_pre']:.3f} → {comp['F_post']:.3f}\n")
        f.write(f"  ω rango: [{comp['omega_min_pre']:.2f}, {comp['omega_max_pre']:.2f}] → "
                f"[{comp['omega_min_post']:.2f}, {comp['omega_max_post']:.2f}]\n")
        f.write(f"  Modos localizados: {comp['n_localized_pre']} → {comp['n_localized_post']}\n")
        f.write(f"  Mobility edge: {comp['mobility_edge_pre']} → {comp['mobility_edge_post']}\n")
    
    f.write("\n=== Resumen por N ===\n")
    for N in Ns_done:
        s = summary_per_N[N]
        f.write(f"\nN={N} ({s['n_replicas']} réplicas):\n")
        f.write(f"  σ(F_local): {s['F_std_mean']:.4f} ± {s['F_std_se']:.4f}\n")
        f.write(f"  F_global: {s['F_global_init_mean']:.2f} → {s['F_global_final_mean']:.2f}\n")
        f.write(f"  ξ_low={s['xi_def_low_mean']}, ξ_high={s['xi_def_high_mean']}\n")
        f.write(f"  Mobility edge ω: {s['mobility_edge_mean']}\n")
        f.write(f"  Enr low: {s['enrichment_low_mean']}, Enr high: {s['enrichment_high_mean']}\n")
    
    f.write("\n=== Conclusiones ===\n")
    for c in report['conclusiones']:
        f.write(f"  - {c}\n")

log_handle.close()

import shutil
shutil.make_archive('/content/dee_materia_v3', 'zip', OUTPUT_DIR)
print("\nResultados empaquetados.")

try:
    from google.colab import files
    files.download('/content/dee_materia_v3.zip')
    print("Descarga iniciada.")
except ImportError:
    print("Descarga manual desde /content/dee_materia_v3.zip")

print("\n" + "="*70)
print("COMPLETADO")
print("="*70)